# Customer-Level Behavioural Profile

The transaction-level Gold dataset is aggregated to one record per customer.

The segmentation variables describe customer transaction frequency, monetary exposure, transaction characteristics, recency and behavioural diversity.

Default outcomes and credit score are excluded from the clustering inputs. They will be used only after clustering to profile the risk characteristics of the resulting customer segments.

The resulting customer-level dataset will serve as the foundation for K-Means segmentation.

In [0]:
from pyspark.sql import functions as F

# Load canonical Gold dataset
GOLD_PATH = "/Volumes/workspace/default/bnpl_raw/gold_bnpl"

gold_df = spark.read.format("delta").load(GOLD_PATH)

print("=" * 70)
print("GOLD DATASET")
print("=" * 70)

print(f"Transactions: {gold_df.count():,}")
print(f"Customers: {gold_df.select('customer_id').distinct().count():,}")


# Determine the end date of the portfolio
portfolio_end_date = (
    gold_df
    .agg(F.max("purchase_date").alias("max_purchase_date"))
    .collect()[0]["max_purchase_date"]
)

print(f"Portfolio end date: {portfolio_end_date}")


# ------------------------------------------------------------
# Customer-level behavioural aggregation
# ------------------------------------------------------------

customer_profile = (
    gold_df
    .groupBy("customer_id")
    .agg(

        # Transaction frequency
        F.count("*").alias("transaction_count"),

        # Monetary exposure
        F.sum("principal_ngn").alias("total_exposure_ngn"),
        F.avg("principal_ngn").alias("avg_transaction_ngn"),
        F.max("principal_ngn").alias("max_transaction_ngn"),
        F.stddev("principal_ngn").alias("std_transaction_ngn"),

        # Repayment structure
        F.avg("tenor_days").alias("avg_tenor_days"),
        F.avg("num_installments").alias("avg_installments"),
        F.avg("interest_rate_monthly").alias("avg_interest_rate"),

        # Customer activity period
        F.min("purchase_date").alias("first_purchase_date"),
        F.max("purchase_date").alias("last_purchase_date"),

        # Transaction spacing
        F.avg(
            F.when(
                F.col("days_since_previous_transaction") >= 0,
                F.col("days_since_previous_transaction")
            )
        ).alias("avg_days_between_transactions"),

        # Behavioural diversity
        F.countDistinct("merchant_category").alias(
            "merchant_category_count"
        ),
        F.countDistinct("provider").alias(
            "provider_count"
        ),
        F.countDistinct("customer_state").alias(
            "state_count"
        )
    )
)


# ------------------------------------------------------------
# Derived customer-level behavioural measures
# ------------------------------------------------------------

customer_profile = (
    customer_profile

    # How recently the customer transacted
    .withColumn(
        "days_since_last_transaction",
        F.datediff(
            F.lit(portfolio_end_date),
            F.col("last_purchase_date")
        )
    )

    # Length of observed customer activity
    .withColumn(
        "customer_active_days",
        F.datediff(
            F.col("last_purchase_date"),
            F.col("first_purchase_date")
        )
    )

    # Single-transaction customers have no within-customer
    # standard deviation or transaction gap
    .fillna({
        "std_transaction_ngn": 0.0,
        "avg_days_between_transactions": 0.0
    })
)


# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

customer_count = customer_profile.count()

print("\n" + "=" * 70)
print("CUSTOMER PROFILE CREATED")
print("=" * 70)

print(f"Customer rows: {customer_count:,}")
print(f"Customer columns: {len(customer_profile.columns)}")

print("\nColumns:")
for column in customer_profile.columns:
    print(f" - {column}")


# Check that customer_id is unique
duplicate_customer_ids = (
    customer_profile
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\n" + "=" * 70)
print("UNIQUENESS CHECK")
print("=" * 70)

print(f"Duplicate customer IDs: {duplicate_customer_ids}")


# ------------------------------------------------------------
# Null check
# ------------------------------------------------------------

profile_columns = [
    "transaction_count",
    "total_exposure_ngn",
    "avg_transaction_ngn",
    "max_transaction_ngn",
    "std_transaction_ngn",
    "avg_tenor_days",
    "avg_installments",
    "avg_interest_rate",
    "avg_days_between_transactions",
    "merchant_category_count",
    "provider_count",
    "state_count",
    "days_since_last_transaction",
    "customer_active_days"
]

print("\n" + "=" * 70)
print("NULL CHECK")
print("=" * 70)

null_check = customer_profile.select([
    F.sum(
        F.when(F.col(column).isNull(), 1).otherwise(0)
    ).alias(column)
    for column in profile_columns
])

null_check.show(truncate=False)


# ------------------------------------------------------------
# Descriptive statistics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CUSTOMER PROFILE DESCRIPTIVE STATISTICS")
print("=" * 70)

customer_profile.select(profile_columns).describe().show(
    truncate=False
)


# ------------------------------------------------------------
# Save customer-level segmentation base
# ------------------------------------------------------------

CUSTOMER_PROFILE_PATH = (
    "/Volumes/workspace/default/bnpl_raw/customer_segmentation_base"
)

(
    customer_profile
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(CUSTOMER_PROFILE_PATH)
)

print("\n" + "=" * 70)
print("CUSTOMER SEGMENTATION BASE SAVED")
print("=" * 70)

print(CUSTOMER_PROFILE_PATH)
print("=" * 70)

GOLD DATASET
Transactions: 2,000,000
Customers: 633,356
Portfolio end date: 2024-12-30 00:00:00

CUSTOMER PROFILE CREATED
Customer rows: 633,356
Customer columns: 17

Columns:
 - customer_id
 - transaction_count
 - total_exposure_ngn
 - avg_transaction_ngn
 - max_transaction_ngn
 - std_transaction_ngn
 - avg_tenor_days
 - avg_installments
 - avg_interest_rate
 - first_purchase_date
 - last_purchase_date
 - avg_days_between_transactions
 - merchant_category_count
 - provider_count
 - state_count
 - days_since_last_transaction
 - customer_active_days

UNIQUENESS CHECK
Duplicate customer IDs: 0

NULL CHECK
+-----------------+------------------+-------------------+-------------------+-------------------+--------------+----------------+-----------------+-----------------------------+-----------------------+--------------+-----------+---------------------------+--------------------+
|transaction_count|total_exposure_ngn|avg_transaction_ngn|max_transaction_ngn|std_transaction_ngn|avg_tenor_da

## Feature Diagnostics and Preparation

K-Means clustering is distance-based, so variables with large numerical ranges or strong right-skew can disproportionately influence cluster formation.

The customer-level behavioural variables are therefore evaluated for skewness and redundancy before modelling.

Highly skewed monetary and activity variables will be transformed using `log1p`, while variables with naturally bounded ranges will remain on their original scale.

All selected clustering variables will then be standardized so that no variable dominates the distance calculation solely because of its measurement scale.

Default outcomes and credit score remain excluded from the clustering inputs.

In [0]:
from pyspark.sql import functions as F

# Load the saved customer segmentation base
CUSTOMER_PROFILE_PATH = (
    "/Volumes/workspace/default/bnpl_raw/customer_segmentation_base"
)

customer_profile = (
    spark.read
    .format("delta")
    .load(CUSTOMER_PROFILE_PATH)
)

print("=" * 70)
print("CUSTOMER SEGMENTATION BASE LOADED")
print("=" * 70)

print(f"Customers: {customer_profile.count():,}")


# ------------------------------------------------------------
# Variables considered for clustering
# ------------------------------------------------------------

candidate_features = [
    "transaction_count",
    "total_exposure_ngn",
    "avg_transaction_ngn",
    "max_transaction_ngn",
    "std_transaction_ngn",
    "avg_tenor_days",
    "avg_installments",
    "avg_interest_rate",
    "avg_days_between_transactions",
    "merchant_category_count",
    "provider_count",
    "state_count",
    "days_since_last_transaction",
    "customer_active_days"
]


# ------------------------------------------------------------
# Measure skewness
# ------------------------------------------------------------

skewness_results = []

for feature in candidate_features:
    value = (
        customer_profile
        .select(F.skewness(F.col(feature)).alias("skewness"))
        .collect()[0]["skewness"]
    )

    skewness_results.append((feature, float(value)))


skewness_df = spark.createDataFrame(
    skewness_results,
    ["feature", "skewness"]
).orderBy(
    F.abs(F.col("skewness")).desc()
)


print("\n" + "=" * 70)
print("FEATURE SKEWNESS")
print("=" * 70)

skewness_df.show(
    truncate=False,
    n=len(candidate_features)
)


# ------------------------------------------------------------
# Create transformed features
#
# log1p is used because it handles zero values safely.
# ------------------------------------------------------------

customer_features = (
    customer_profile

    # Activity / frequency
    .withColumn(
        "log_transaction_count",
        F.log1p("transaction_count")
    )

    # Monetary variables
    .withColumn(
        "log_total_exposure",
        F.log1p("total_exposure_ngn")
    )
    .withColumn(
        "log_avg_transaction",
        F.log1p("avg_transaction_ngn")
    )
    .withColumn(
        "log_max_transaction",
        F.log1p("max_transaction_ngn")
    )
    .withColumn(
        "log_std_transaction",
        F.log1p("std_transaction_ngn")
    )

    # Transaction spacing
    .withColumn(
        "log_avg_days_between",
        F.log1p("avg_days_between_transactions")
    )

    # Recency
    .withColumn(
        "log_days_since_last",
        F.log1p("days_since_last_transaction")
    )

    # Active duration
    .withColumn(
        "log_customer_active_days",
        F.log1p("customer_active_days")
    )
)


# ------------------------------------------------------------
# Define the clustering feature set
#
# We retain behavioural dimensions while avoiding unnecessary
# duplication of highly similar variables.
# ------------------------------------------------------------

clustering_features = [
    "log_transaction_count",
    "log_total_exposure",
    "log_avg_transaction",
    "log_max_transaction",
    "avg_tenor_days",
    "avg_installments",
    "avg_interest_rate",
    "log_avg_days_between",
    "merchant_category_count",
    "provider_count",
    "state_count",
    "log_days_since_last",
    "log_customer_active_days"
]


print("\n" + "=" * 70)
print("CLUSTERING FEATURES")
print("=" * 70)

for feature in clustering_features:
    print(f" - {feature}")

print(f"\nNumber of clustering variables: {len(clustering_features)}")


# ------------------------------------------------------------
# Check transformed variables
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRANSFORMED FEATURE SUMMARY")
print("=" * 70)

customer_features.select(
    clustering_features
).describe().show(
    truncate=False
)


# ------------------------------------------------------------
# Verify all clustering inputs are valid
# ------------------------------------------------------------

invalid_check = customer_features.select([
    F.sum(
        F.when(
            F.col(feature).isNull() |
            F.isnan(F.col(feature)),
            1
        ).otherwise(0)
    ).alias(feature)
    for feature in clustering_features
])

print("\n" + "=" * 70)
print("INVALID VALUE CHECK")
print("=" * 70)

invalid_check.show(truncate=False)

CUSTOMER SEGMENTATION BASE LOADED
Customers: 633,356

FEATURE SKEWNESS
+-----------------------------+--------------------+
|feature                      |skewness            |
+-----------------------------+--------------------+
|avg_transaction_ngn          |2.813493496691828   |
|std_transaction_ngn          |2.5513742106040427  |
|max_transaction_ngn          |2.2552834541738025  |
|total_exposure_ngn           |1.4458271768545472  |
|avg_days_between_transactions|1.3297411032633406  |
|avg_installments             |1.0794390866869605  |
|days_since_last_transaction  |0.9939803708115463  |
|transaction_count            |0.7682678777134674  |
|avg_tenor_days               |0.7496371506941943  |
|state_count                  |0.6867059190654636  |
|avg_interest_rate            |-0.4819975989442848 |
|merchant_category_count      |0.3838137604617398  |
|provider_count               |0.37458976825575563 |
|customer_active_days         |-0.09705643123679987|
+---------------------------

## Final Feature Selection and Standardization

The transformed behavioural variables are reviewed for excessive correlation before clustering.

Highly redundant variables can cause a single behavioural dimension to receive disproportionate influence in distance calculations. The final feature set therefore balances monetary exposure, transaction frequency, recency, transaction structure and behavioural diversity.

The selected variables are standardized using Spark ML's `StandardScaler`. This ensures that differences in measurement units do not determine cluster formation.

The clustering dataset contains no credit-score or default-derived variables.

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Load the customer-level segmentation base
CUSTOMER_PROFILE_PATH = (
    "/Volumes/workspace/default/bnpl_raw/customer_segmentation_base"
)

customer_profile = (
    spark.read
    .format("delta")
    .load(CUSTOMER_PROFILE_PATH)
)


# ------------------------------------------------------------
# Recreate transformed behavioural variables
# ------------------------------------------------------------

customer_features = (
    customer_profile

    .withColumn(
        "log_transaction_count",
        F.log1p("transaction_count")
    )

    .withColumn(
        "log_total_exposure",
        F.log1p("total_exposure_ngn")
    )

    .withColumn(
        "log_avg_transaction",
        F.log1p("avg_transaction_ngn")
    )

    .withColumn(
        "log_max_transaction",
        F.log1p("max_transaction_ngn")
    )

    .withColumn(
        "log_std_transaction",
        F.log1p("std_transaction_ngn")
    )

    .withColumn(
        "log_avg_days_between",
        F.log1p("avg_days_between_transactions")
    )

    .withColumn(
        "log_days_since_last",
        F.log1p("days_since_last_transaction")
    )
)


# ------------------------------------------------------------
# Final candidate clustering features
#
# customer_active_days is intentionally kept on its original
# scale because its original skewness was approximately -0.10.
# ------------------------------------------------------------

clustering_features = [
    "log_transaction_count",
    "log_total_exposure",
    "log_avg_transaction",
    "log_max_transaction",
    "log_std_transaction",
    "avg_tenor_days",
    "avg_installments",
    "avg_interest_rate",
    "log_avg_days_between",
    "merchant_category_count",
    "provider_count",
    "state_count",
    "log_days_since_last",
    "customer_active_days"
]


# ------------------------------------------------------------
# Correlation diagnostics
# ------------------------------------------------------------

print("=" * 70)
print("CORRELATION DIAGNOSTIC")
print("=" * 70)

correlation_results = []

for i in range(len(clustering_features)):
    for j in range(i + 1, len(clustering_features)):

        feature_1 = clustering_features[i]
        feature_2 = clustering_features[j]

        correlation = customer_features.stat.corr(
            feature_1,
            feature_2
        )

        correlation_results.append(
            (
                feature_1,
                feature_2,
                float(correlation)
            )
        )


correlation_df = (
    spark.createDataFrame(
        correlation_results,
        ["feature_1", "feature_2", "correlation"]
    )
    .withColumn(
        "absolute_correlation",
        F.abs(F.col("correlation"))
    )
    .orderBy(
        F.col("absolute_correlation").desc()
    )
)

print("\nTop feature correlations:")

correlation_df.show(
    20,
    truncate=False
)


# ------------------------------------------------------------
# Flag highly correlated pairs
# ------------------------------------------------------------

high_correlation_df = (
    correlation_df
    .filter(F.col("absolute_correlation") >= 0.80)
)

print("\n" + "=" * 70)
print("HIGH-CORRELATION PAIRS | |r| >= 0.80")
print("=" * 70)

high_correlation_df.show(
    truncate=False
)


# ------------------------------------------------------------
# Assemble clustering features
# ------------------------------------------------------------

assembler = VectorAssembler(
    inputCols=clustering_features,
    outputCol="clustering_features_raw"
)

assembled_df = assembler.transform(customer_features)


# ------------------------------------------------------------
# Standardize features
# ------------------------------------------------------------

scaler = StandardScaler(
    inputCol="clustering_features_raw",
    outputCol="features",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(assembled_df)

scaled_df = scaler_model.transform(assembled_df)


# ------------------------------------------------------------
# Verify the clustering vector
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STANDARDIZATION COMPLETE")
print("=" * 70)

print(f"Customers: {scaled_df.count():,}")
print(f"Clustering variables: {len(clustering_features)}")

scaled_df.select(
    "customer_id",
    "features"
).show(
    5,
    truncate=False
)


# ------------------------------------------------------------
# Save the prepared clustering dataset
# ------------------------------------------------------------

CLUSTERING_BASE_PATH = (
    "/Volumes/workspace/default/bnpl_raw/customer_clustering_features"
)

(
    scaled_df
    .select(
        "customer_id",
        *clustering_features,
        "features"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(CLUSTERING_BASE_PATH)
)

print("\n" + "=" * 70)
print("CLUSTERING FEATURE DATASET SAVED")
print("=" * 70)

print(CLUSTERING_BASE_PATH)

CORRELATION DIAGNOSTIC

Top feature correlations:
+-----------------------+-----------------------+------------------+--------------------+
|feature_1              |feature_2              |correlation       |absolute_correlation|
+-----------------------+-----------------------+------------------+--------------------+
|avg_tenor_days         |avg_installments       |0.9650530303861882|0.9650530303861882  |
|log_std_transaction    |log_avg_days_between   |0.9164553687124541|0.9164553687124541  |
|log_transaction_count  |state_count            |0.9074651079619493|0.9074651079619493  |
|log_avg_transaction    |log_max_transaction    |0.9033916497552985|0.9033916497552985  |
|log_total_exposure     |log_max_transaction    |0.8937880044472072|0.8937880044472072  |
|log_transaction_count  |provider_count         |0.8225360238108246|0.8225360238108246  |
|log_transaction_count  |merchant_category_count|0.7881068140858322|0.7881068140858322  |
|log_transaction_count  |log_total_exposure     |0

## Final Clustering Feature Set

The correlation analysis identified several highly redundant variables.

To reduce duplication in the distance calculation, the final clustering representation retains nine behavioural dimensions:

- transaction frequency
- total exposure
- average transaction size
- average tenor
- average interest rate
- average transaction spacing
- merchant-category diversity
- customer recency
- customer active duration

Highly correlated variables representing similar information were excluded, including maximum transaction size, transaction-level standard deviation, provider diversity, state diversity and average installment count.

This feature set is intended to provide a balanced representation of customer activity, exposure, transaction structure and engagement without allowing closely related variables to disproportionately influence cluster formation.

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Load customer-level profile
CUSTOMER_PROFILE_PATH = (
    "/Volumes/workspace/default/bnpl_raw/customer_segmentation_base"
)

customer_profile = (
    spark.read
    .format("delta")
    .load(CUSTOMER_PROFILE_PATH)
)


# ------------------------------------------------------------
# Create transformed variables
# ------------------------------------------------------------

customer_features = (
    customer_profile

    .withColumn(
        "log_transaction_count",
        F.log1p("transaction_count")
    )

    .withColumn(
        "log_total_exposure",
        F.log1p("total_exposure_ngn")
    )

    .withColumn(
        "log_avg_transaction",
        F.log1p("avg_transaction_ngn")
    )

    .withColumn(
        "log_avg_days_between",
        F.log1p("avg_days_between_transactions")
    )

    .withColumn(
        "log_days_since_last",
        F.log1p("days_since_last_transaction")
    )
)


# ------------------------------------------------------------
# Final clustering variables
# ------------------------------------------------------------

clustering_features = [
    "log_transaction_count",
    "log_total_exposure",
    "log_avg_transaction",
    "avg_tenor_days",
    "avg_interest_rate",
    "log_avg_days_between",
    "merchant_category_count",
    "log_days_since_last",
    "customer_active_days"
]


print("=" * 70)
print("FINAL CLUSTERING FEATURE SET")
print("=" * 70)

for feature in clustering_features:
    print(f" - {feature}")

print(f"\nNumber of clustering variables: {len(clustering_features)}")


# ------------------------------------------------------------
# Assemble feature vector
# ------------------------------------------------------------

assembler = VectorAssembler(
    inputCols=clustering_features,
    outputCol="clustering_features_raw"
)

assembled_df = assembler.transform(customer_features)


# ------------------------------------------------------------
# Standardize
# ------------------------------------------------------------

scaler = StandardScaler(
    inputCol="clustering_features_raw",
    outputCol="features",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(assembled_df)

scaled_df = scaler_model.transform(assembled_df)


# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STANDARDIZATION COMPLETE")
print("=" * 70)

print(f"Customers: {scaled_df.count():,}")
print(f"Features: {len(clustering_features)}")

scaled_df.select(
    "customer_id",
    "features"
).show(
    5,
    truncate=False
)


# ------------------------------------------------------------
# Save prepared clustering dataset
# ------------------------------------------------------------

CLUSTERING_BASE_PATH = (
    "/Volumes/workspace/default/bnpl_raw/customer_clustering_features"
)

(
    scaled_df
    .select(
        "customer_id",
        *clustering_features,
        "features"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(CLUSTERING_BASE_PATH)
)


print("\n" + "=" * 70)
print("CLUSTERING FEATURE DATASET SAVED")
print("=" * 70)

print(CLUSTERING_BASE_PATH)

FINAL CLUSTERING FEATURE SET
 - log_transaction_count
 - log_total_exposure
 - log_avg_transaction
 - avg_tenor_days
 - avg_interest_rate
 - log_avg_days_between
 - merchant_category_count
 - log_days_since_last
 - customer_active_days

Number of clustering variables: 9

STANDARDIZATION COMPLETE
Customers: 633,356
Features: 9
+------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|customer_id |features                                                                                                                                                                          |
+------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|CUS-00001762|[0.09609502754078847,-0.5563906357278855,-1.0158903560207067,0.2879507371743

## Selecting the Number of Customer Segments

The number of customer segments is selected empirically rather than being predetermined.

K-Means models with different values of K are evaluated using the Silhouette Score, which measures how well each customer fits within its assigned cluster relative to neighbouring clusters.

A representative customer sample is used for model selection to reduce computational cost while preserving the overall structure of the portfolio.

The final K will be selected by considering both clustering quality and interpretability. The selected value will then be used to train the final K-Means model across the full customer population.

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Load standardized clustering dataset
CLUSTERING_BASE_PATH = (
    "/Volumes/workspace/default/bnpl_raw/customer_clustering_features"
)

clustering_df = (
    spark.read
    .format("delta")
    .load(CLUSTERING_BASE_PATH)
)

print("=" * 70)
print("CLUSTERING DATASET")
print("=" * 70)

total_customers = clustering_df.count()

print(f"Total customers: {total_customers:,}")


# ------------------------------------------------------------
# Create a fixed representative sample
# ------------------------------------------------------------

SAMPLE_FRACTION = 0.10
RANDOM_SEED = 42

clustering_sample = (
    clustering_df
    .sample(
        withReplacement=False,
        fraction=SAMPLE_FRACTION,
        seed=RANDOM_SEED
    )
)

sample_count = clustering_sample.count()

print(f"Sample customers: {sample_count:,}")
print(f"Sample proportion: {sample_count / total_customers:.2%}")


# ------------------------------------------------------------
# Evaluate candidate K values
# ------------------------------------------------------------

evaluator = ClusteringEvaluator(
    predictionCol="prediction",
    featuresCol="features",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

k_values = [2, 3, 4, 5, 6, 7, 8]

results = []

print("\n" + "=" * 70)
print("K-MEANS MODEL SELECTION")
print("=" * 70)

for k in k_values:

    print(f"\nEvaluating K = {k}...")

    kmeans = (
        KMeans(
            k=k,
            seed=RANDOM_SEED,
            featuresCol="features",
            predictionCol="prediction",
            maxIter=30
        )
    )

    model = kmeans.fit(clustering_sample)

    predictions = model.transform(clustering_sample)

    silhouette = evaluator.evaluate(predictions)

    results.append(
        (
            k,
            float(silhouette)
        )
    )

    print(f"Silhouette Score: {silhouette:.4f}")


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

results_df = (
    spark.createDataFrame(
        results,
        ["k", "silhouette_score"]
    )
    .orderBy("k")
)

print("\n" + "=" * 70)
print("SILHOUETTE SCORE RESULTS")
print("=" * 70)

results_df.show(
    truncate=False
)


# ------------------------------------------------------------
# Identify highest-scoring K
# ------------------------------------------------------------

best_k_row = (
    results_df
    .orderBy(F.col("silhouette_score").desc())
    .first()
)

best_k = int(best_k_row["k"])
best_silhouette = float(best_k_row["silhouette_score"])

print("\n" + "=" * 70)
print("HIGHEST SILHOUETTE SCORE")
print("=" * 70)

print(f"Best K by Silhouette Score: {best_k}")
print(f"Silhouette Score: {best_silhouette:.4f}")

CLUSTERING DATASET
Total customers: 633,356
Sample customers: 63,152
Sample proportion: 9.97%

K-MEANS MODEL SELECTION

Evaluating K = 2...
Silhouette Score: 0.5329

Evaluating K = 3...
Silhouette Score: 0.3323

Evaluating K = 4...
Silhouette Score: 0.2826

Evaluating K = 5...
Silhouette Score: 0.2611

Evaluating K = 6...
Silhouette Score: 0.2202

Evaluating K = 7...
Silhouette Score: 0.2421

Evaluating K = 8...
Silhouette Score: 0.2490

SILHOUETTE SCORE RESULTS
+---+-------------------+
|k  |silhouette_score   |
+---+-------------------+
|2  |0.5328505323535742 |
|3  |0.3323376223945642 |
|4  |0.28261047063101813|
|5  |0.26107359223594057|
|6  |0.2201925165246652 |
|7  |0.2421239755042609 |
|8  |0.2490152050590651 |
+---+-------------------+


HIGHEST SILHOUETTE SCORE
Best K by Silhouette Score: 2
Silhouette Score: 0.5329


In [0]:
# ------------------------------------------------------------
# K=2 RANDOM SEED ROBUSTNESS CHECK
# ------------------------------------------------------------

seed_results = []

test_seeds = [7, 21, 42]

print("\n" + "=" * 70)
print("K=2 RANDOM SEED ROBUSTNESS CHECK")
print("=" * 70)

for seed in test_seeds:

    kmeans_seed_test = KMeans(
        k=2,
        seed=seed,
        featuresCol="features",
        predictionCol="prediction",
        maxIter=30
    )

    seed_model = kmeans_seed_test.fit(
        clustering_sample
    )

    seed_predictions = seed_model.transform(
        clustering_sample
    )

    seed_silhouette = evaluator.evaluate(
        seed_predictions
    )

    seed_results.append(
        (
            seed,
            float(seed_silhouette)
        )
    )

    print(
        f"Seed {seed}: "
        f"Silhouette Score = {seed_silhouette:.4f}"
    )


seed_results_df = (
    spark.createDataFrame(
        seed_results,
        ["seed", "silhouette_score"]
    )
    .orderBy("seed")
)

print("\n" + "=" * 70)
print("RANDOM SEED RESULTS")
print("=" * 70)

seed_results_df.show(
    truncate=False
)


K=2 RANDOM SEED ROBUSTNESS CHECK
Seed 7: Silhouette Score = 0.5331
Seed 21: Silhouette Score = 0.5329
Seed 42: Silhouette Score = 0.5329

RANDOM SEED RESULTS
+----+------------------+
|seed|silhouette_score  |
+----+------------------+
|7   |0.533143667371594 |
|21  |0.5328505323535742|
|42  |0.5328505323535742|
+----+------------------+



## Final K-Means Clustering

The Silhouette Score was evaluated for K values from 2 to 8.

K=2 produced the highest Silhouette Score of 0.532, substantially exceeding the scores obtained for all higher values of K. This indicates that the customer population is most clearly separated into two behavioural groups within the selected feature space.

The final K-Means model therefore uses two clusters and is fitted across the full population of 633,356 customers.

Cluster labels are treated as arbitrary identifiers. Their business interpretation will be determined only after examining the behavioural characteristics of each cluster.

### K=2 Random Seed Robustness

The K=2 clustering solution was tested using multiple random seeds while keeping the clustering sample, feature set and model configuration unchanged. The resulting Silhouette Scores were broadly consistent across seeds, indicating that the identified two-cluster structure is not materially dependent on a particular random initialization.

The original seed of 42 is therefore retained for the final K-Means model. This provides additional robustness support for the selected K=2 solution without altering the final segmentation methodology.

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.clustering import KMeans

# Load the standardized clustering dataset
CLUSTERING_BASE_PATH = (
    "/Volumes/workspace/default/bnpl_raw/customer_clustering_features"
)

clustering_df = (
    spark.read
    .format("delta")
    .load(CLUSTERING_BASE_PATH)
)

print("=" * 70)
print("FINAL K-MEANS MODEL")
print("=" * 70)

print(f"Customers: {clustering_df.count():,}")


# ------------------------------------------------------------
# Fit final K-Means model
# ------------------------------------------------------------

FINAL_K = 2
RANDOM_SEED = 42

kmeans = KMeans(
    k=FINAL_K,
    seed=RANDOM_SEED,
    featuresCol="features",
    predictionCol="cluster",
    maxIter=50
)

kmeans_model = kmeans.fit(clustering_df)

clustered_df = kmeans_model.transform(clustering_df)


# ------------------------------------------------------------
# Cluster sizes
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLUSTER SIZES")
print("=" * 70)

cluster_sizes = (
    clustered_df
    .groupBy("cluster")
    .agg(
        F.count("*").alias("customers")
    )
    .withColumn(
        "portfolio_share_pct",
        F.round(
            F.col("customers") /
            F.lit(clustering_df.count()) * 100,
            2
        )
    )
    .orderBy("cluster")
)

cluster_sizes.show(truncate=False)


# ------------------------------------------------------------
# Cluster centres
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLUSTER CENTRES")
print("=" * 70)

centers = kmeans_model.clusterCenters()

for i, center in enumerate(centers):
    print(f"\nCluster {i}:")
    for feature, value in zip(
        [
            "log_transaction_count",
            "log_total_exposure",
            "log_avg_transaction",
            "avg_tenor_days",
            "avg_interest_rate",
            "log_avg_days_between",
            "merchant_category_count",
            "log_days_since_last",
            "customer_active_days"
        ],
        center
    ):
        print(f"  {feature}: {value:.4f}")


# ------------------------------------------------------------
# Save initial clustered customer dataset
# ------------------------------------------------------------

CLUSTERED_CUSTOMERS_PATH = (
    "/Volumes/workspace/default/bnpl_raw/customer_kmeans_clusters"
)

(
    clustered_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(CLUSTERED_CUSTOMERS_PATH)
)

print("\n" + "=" * 70)
print("CLUSTERED CUSTOMER DATASET SAVED")
print("=" * 70)

print(CLUSTERED_CUSTOMERS_PATH)

FINAL K-MEANS MODEL
Customers: 633,356

CLUSTER SIZES
+-------+---------+-------------------+
|cluster|customers|portfolio_share_pct|
+-------+---------+-------------------+
|0      |150376   |23.74              |
|1      |482980   |76.26              |
+-------+---------+-------------------+


CLUSTER CENTRES

Cluster 0:
  log_transaction_count: -1.2920
  log_total_exposure: -1.1835
  log_avg_transaction: -0.4290
  avg_tenor_days: -0.0016
  avg_interest_rate: -0.0025
  log_avg_days_between: -1.4573
  merchant_category_count: -1.0773
  log_days_since_last: 0.6588
  customer_active_days: -1.3189

Cluster 1:
  log_transaction_count: 0.4023
  log_total_exposure: 0.3685
  log_avg_transaction: 0.1336
  avg_tenor_days: 0.0005
  avg_interest_rate: 0.0008
  log_avg_days_between: 0.4537
  merchant_category_count: 0.3354
  log_days_since_last: -0.2051
  customer_active_days: 0.4107

CLUSTERED CUSTOMER DATASET SAVED
/Volumes/workspace/default/bnpl_raw/customer_kmeans_clusters


## Risk Profiling of Customer Segments

The K-Means clusters were created exclusively from behavioural, exposure and engagement variables.

Default outcomes and credit score are now reintroduced only for post-clustering profiling. This allows the analysis to determine whether the behaviourally defined customer segments exhibit different observed credit-risk characteristics without using risk outcomes to construct the clusters.

The analysis compares default incidence, exposure and credit characteristics across the resulting customer segments.

These comparisons describe patterns within the synthetic BNPL dataset and should not be interpreted as empirical evidence about the Nigerian BNPL market.

In [0]:
from pyspark.sql import functions as F

# Load the customer cluster assignments
CLUSTERED_CUSTOMERS_PATH = (
    "/Volumes/workspace/default/bnpl_raw/customer_kmeans_clusters"
)

clustered_customers = (
    spark.read
    .format("delta")
    .load(CLUSTERED_CUSTOMERS_PATH)
)


# Load the original Gold transaction-level dataset
GOLD_PATH = "/Volumes/workspace/default/bnpl_raw/gold_bnpl"

gold_df = (
    spark.read
    .format("delta")
    .load(GOLD_PATH)
)


# ------------------------------------------------------------
# Join customer clusters back to transaction-level data
# ------------------------------------------------------------

clustered_transactions = (
    gold_df
    .join(
        clustered_customers.select(
            "customer_id",
            "cluster"
        ),
        on="customer_id",
        how="inner"
    )
)


# ------------------------------------------------------------
# Profile each customer segment
# ------------------------------------------------------------

cluster_risk_profile = (
    clustered_transactions
    .groupBy("cluster")
    .agg(

        # Customer population
        F.countDistinct("customer_id").alias(
            "customers"
        ),

        # Transaction volume
        F.count("*").alias(
            "transactions"
        ),

        # Exposure
        F.sum("principal_ngn").alias(
            "total_exposure_ngn"
        ),

        F.avg("principal_ngn").alias(
            "avg_transaction_ngn"
        ),

        # Default outcomes
        F.sum(
            F.col("default_30d").cast("long")
        ).alias(
            "default_30d_count"
        ),

        F.sum(
            F.col("default_90d").cast("long")
        ).alias(
            "default_90d_count"
        ),

        # Credit score, used only for profiling
        F.avg("credit_score").alias(
            "avg_credit_score"
        ),

        # First-time customer share, used only for profiling
        F.avg(
            F.col("first_time_customer").cast("double")
        ).alias(
            "first_time_customer_rate"
        )
    )
)


# ------------------------------------------------------------
# Derive rates and portfolio shares
# ------------------------------------------------------------

total_transactions = gold_df.count()

total_exposure = (
    gold_df
    .agg(F.sum("principal_ngn"))
    .collect()[0][0]
)

cluster_risk_profile = (
    cluster_risk_profile

    .withColumn(
        "portfolio_share_pct",
        F.round(
            F.col("transactions") /
            F.lit(total_transactions) * 100,
            2
        )
    )

    .withColumn(
        "exposure_share_pct",
        F.round(
            F.col("total_exposure_ngn") /
            F.lit(total_exposure) * 100,
            2
        )
    )

    .withColumn(
        "default_30d_rate_pct",
        F.round(
            F.col("default_30d_count") /
            F.col("transactions") * 100,
            2
        )
    )

    .withColumn(
        "default_90d_rate_pct",
        F.round(
            F.col("default_90d_count") /
            F.col("transactions") * 100,
            2
        )
    )

    .withColumn(
        "avg_exposure_per_customer_ngn",
        F.round(
            F.col("total_exposure_ngn") /
            F.col("customers"),
            2
        )
    )

    .withColumn(
        "avg_transactions_per_customer",
        F.round(
            F.col("transactions") /
            F.col("customers"),
            2
        )
    )

    .withColumn(
        "avg_credit_score",
        F.round(
            F.col("avg_credit_score"),
            2
        )
    )

    .withColumn(
        "first_time_customer_rate_pct",
        F.round(
            F.col("first_time_customer_rate") * 100,
            2
        )
    )
)


# ------------------------------------------------------------
# Display risk profile
# ------------------------------------------------------------

print("=" * 75)
print("CUSTOMER SEGMENT RISK PROFILE")
print("=" * 75)

cluster_risk_profile.select(
    "cluster",
    "customers",
    "transactions",
    "portfolio_share_pct",
    "total_exposure_ngn",
    "exposure_share_pct",
    "avg_transaction_ngn",
    "avg_exposure_per_customer_ngn",
    "avg_transactions_per_customer",
    "default_30d_count",
    "default_30d_rate_pct",
    "default_90d_count",
    "default_90d_rate_pct",
    "avg_credit_score",
    "first_time_customer_rate_pct"
).orderBy("cluster").show(
    truncate=False
)

CUSTOMER SEGMENT RISK PROFILE
+-------+---------+------------+-------------------+-------------------+------------------+-------------------+-----------------------------+-----------------------------+-----------------+--------------------+-----------------+--------------------+----------------+----------------------------+
|cluster|customers|transactions|portfolio_share_pct|total_exposure_ngn |exposure_share_pct|avg_transaction_ngn|avg_exposure_per_customer_ngn|avg_transactions_per_customer|default_30d_count|default_30d_rate_pct|default_90d_count|default_90d_rate_pct|avg_credit_score|first_time_customer_rate_pct|
+-------+---------+------------+-------------------+-------------------+------------------+-------------------+-----------------------------+-----------------------------+-----------------+--------------------+-----------------+--------------------+----------------+----------------------------+
|0      |150376   |204666      |10.23              |8.481038134001303E9|8.48      

## Customer Segment Interpretation

The two K-Means clusters represent distinct behavioural and exposure profiles rather than distinct risk classes.

Cluster 0 is characterized by lower transaction frequency, lower exposure, lower behavioural diversity and shorter observed customer activity. It is therefore classified as the Low-Engagement / Low-Exposure segment.

Cluster 1 contains customers with substantially higher transaction frequency, exposure and behavioural engagement. It is classified as the Active / Higher-Exposure segment.

Post-clustering risk profiling shows only a modest difference in observed default rates between the segments. Therefore, the clusters should not be interpreted as low-risk and high-risk customer groups.

This distinction is important: behavioural segmentation identifies portfolio structure, while the classification model provides transaction-level default discrimination.

In [0]:
from pyspark.sql import functions as F

# Load clustered customer transactions
CLUSTERED_CUSTOMERS_PATH = (
    "/Volumes/workspace/default/bnpl_raw/customer_kmeans_clusters"
)

clustered_customers = (
    spark.read
    .format("delta")
    .load(CLUSTERED_CUSTOMERS_PATH)
)

GOLD_PATH = "/Volumes/workspace/default/bnpl_raw/gold_bnpl"

gold_df = (
    spark.read
    .format("delta")
    .load(GOLD_PATH)
)


# Join cluster assignments to transactions
clustered_transactions = (
    gold_df
    .join(
        clustered_customers.select(
            "customer_id",
            "cluster"
        ),
        on="customer_id",
        how="inner"
    )
)


# Total portfolio defaults
total_30d_defaults = (
    gold_df
    .agg(F.sum(F.col("default_30d").cast("long")))
    .collect()[0][0]
)

total_90d_defaults = (
    gold_df
    .agg(F.sum(F.col("default_90d").cast("long")))
    .collect()[0][0]
)


# ------------------------------------------------------------
# Final segment contribution analysis
# ------------------------------------------------------------

segment_summary = (
    clustered_transactions
    .groupBy("cluster")
    .agg(
        F.countDistinct("customer_id").alias("customers"),
        F.count("*").alias("transactions"),

        F.sum("principal_ngn").alias(
            "total_exposure_ngn"
        ),

        F.sum(
            F.col("default_30d").cast("long")
        ).alias(
            "default_30d_count"
        ),

        F.sum(
            F.col("default_90d").cast("long")
        ).alias(
            "default_90d_count"
        )
    )

    .withColumn(
        "default_30d_rate_pct",
        F.round(
            F.col("default_30d_count") /
            F.col("transactions") * 100,
            2
        )
    )

    .withColumn(
        "default_90d_rate_pct",
        F.round(
            F.col("default_90d_count") /
            F.col("transactions") * 100,
            2
        )
    )

    .withColumn(
        "default_30d_contribution_pct",
        F.round(
            F.col("default_30d_count") /
            F.lit(total_30d_defaults) * 100,
            2
        )
    )

    .withColumn(
        "default_90d_contribution_pct",
        F.round(
            F.col("default_90d_count") /
            F.lit(total_90d_defaults) * 100,
            2
        )
    )

    .withColumn(
        "segment_label",
        F.when(
            F.col("cluster") == 0,
            "Low-Engagement / Low-Exposure"
        )
        .otherwise(
            "Active / Higher-Exposure"
        )
    )
)


print("=" * 75)
print("FINAL CUSTOMER SEGMENT SUMMARY")
print("=" * 75)

segment_summary.select(
    "cluster",
    "segment_label",
    "customers",
    "transactions",
    "total_exposure_ngn",
    "default_30d_rate_pct",
    "default_90d_rate_pct",
    "default_30d_contribution_pct",
    "default_90d_contribution_pct"
).orderBy("cluster").show(
    truncate=False
)


# ------------------------------------------------------------
# Save final segmentation output
# ------------------------------------------------------------

FINAL_SEGMENT_PATH = (
    "/Volumes/workspace/default/bnpl_raw/final_bnpl_customer_segments"
)

(
    segment_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(FINAL_SEGMENT_PATH)
)


print("\n" + "=" * 75)
print("FINAL CUSTOMER SEGMENT SUMMARY SAVED")
print("=" * 75)

print(FINAL_SEGMENT_PATH)

FINAL CUSTOMER SEGMENT SUMMARY
+-------+-----------------------------+---------+------------+-------------------+--------------------+--------------------+----------------------------+----------------------------+
|cluster|segment_label                |customers|transactions|total_exposure_ngn |default_30d_rate_pct|default_90d_rate_pct|default_30d_contribution_pct|default_90d_contribution_pct|
+-------+-----------------------------+---------+------------+-------------------+--------------------+--------------------+----------------------------+----------------------------+
|0      |Low-Engagement / Low-Exposure|150376   |204666      |8.481038134001303E9|4.46                |7.31                |9.13                        |9.35                        |
|1      |Active / Higher-Exposure     |482980   |1795334     |9.14749804860829E10|5.06                |8.08                |90.87                       |90.65                       |
+-------+-----------------------------+---------+-----

In [0]:
# ------------------------------------------------------------
# FINAL CUSTOMER SEGMENTATION QUALITY GATE
# ------------------------------------------------------------

print("=" * 75)
print("FINAL CUSTOMER SEGMENTATION QUALITY GATE")
print("=" * 75)

# Customer-level checks
customer_count = (
    clustered_customers
    .select("customer_id")
    .distinct()
    .count()
)

duplicate_customer_ids = (
    clustered_customers
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_cluster_assignments = (
    clustered_customers
    .filter(F.col("cluster").isNull())
    .count()
)

cluster_count = (
    clustered_customers
    .select("cluster")
    .distinct()
    .count()
)

# Final segment summary checks
segment_count = segment_summary.count()

# Clustering input check
forbidden_features = [
    "default_30d",
    "default_90d",
    "credit_score"
]

forbidden_features_present = [
    feature
    for feature in forbidden_features
    if feature in clustering_df.columns
]

# Quality checks
quality_checks = [
    ("Customer count = 633,356", customer_count == 633356),
    ("No duplicate customer IDs", duplicate_customer_ids == 0),
    ("No null cluster assignments", null_cluster_assignments == 0),
    ("Exactly 2 clusters", cluster_count == 2),
    (
        "Default and credit score excluded from clustering",
        len(forbidden_features_present) == 0
    ),
    ("Final segment summary contains 2 segments", segment_count == 2),
    ("Selected K = 2", best_k == 2)
]

quality_gate_df = spark.createDataFrame(
    quality_checks,
    ["quality_check", "passed"]
)

quality_gate_df.show(
    truncate=False
)

all_checks_passed = all(
    check[1]
    for check in quality_checks
)

print("\n" + "=" * 75)

if all_checks_passed:
    print("FINAL CUSTOMER SEGMENTATION QUALITY GATE: PASSED")
else:
    print("FINAL CUSTOMER SEGMENTATION QUALITY GATE: REVIEW REQUIRED")

print("=" * 75)

FINAL CUSTOMER SEGMENTATION QUALITY GATE
+-------------------------------------------------+------+
|quality_check                                    |passed|
+-------------------------------------------------+------+
|Customer count = 633,356                         |true  |
|No duplicate customer IDs                        |true  |
|No null cluster assignments                      |true  |
|Exactly 2 clusters                               |true  |
|Default and credit score excluded from clustering|true  |
|Final segment summary contains 2 segments        |true  |
|Selected K = 2                                   |true  |
+-------------------------------------------------+------+


FINAL CUSTOMER SEGMENTATION QUALITY GATE: PASSED


## Final Segmentation Quality Gate

The final customer segmentation passed all defined quality checks. The complete customer population of 633,356 customers is represented, customer IDs are unique, cluster assignments are non-null, and exactly two clusters are present.

The clustering feature set excludes default outcomes and credit score, ensuring that the segments are formed from customer behavioural and exposure characteristics rather than directly from credit-risk outcomes.

The final segmentation provides two interpretable customer groups that support portfolio monitoring, exposure concentration analysis and customer strategy. The segments should be interpreted as behavioural and portfolio-structure groups rather than direct credit-risk classes.

All findings are based on the synthetic BNPL dataset and should not be interpreted as empirical evidence about the Nigerian BNPL market.